# Notebook 02 - Embeddings, Tokenizacion y Self-Attention

## Objetivos
- Usar tokenizadores de Hugging Face sobre texto real.
- Relacionar tokens con vectores de embedding.
- Implementar Scaled Dot-Product Attention desde cero con PyTorch.

## Introduccion
La Self-Attention es el nucleo del Transformer. Antes de cargar modelos preentrenados, conviene entender como se calculan Q, K y V y como se obtienen pesos de atencion normalizados.

In [1]:
from pathlib import Path
from IPython.display import display
import torch
import torch.nn.functional as F
import pandas as pd
from transformers import AutoTokenizer

print("PyTorch version:", torch.__version__)

PyTorch version: 2.8.0+cpu


## 1) Tokenizacion con Hugging Face

In [2]:
# Escribe tu codigo aqui
tokenizar = AutoTokenizer.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

c:\Users\juand\anaconda3\envs\tf_windows\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\juand\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [20]:
frase_ejemplo = "Hello world! This is an example sentence for tokenization."

In [21]:
encoding = tokenizar(frase_ejemplo, return_tensors="pt")
print("Token IDs:", encoding["input_ids"])
print("Token:", tokenizar.convert_ids_to_tokens(encoding["input_ids"][0]))
print("input_ids shape:", encoding["input_ids"].shape)

Token IDs: tensor([[  101,  7592,  2088,   999,  2023,  2003,  2019,  2742,  6251,  2005,
         19204,  3989,  1012,   102]])
Token: ['[CLS]', 'hello', 'world', '!', 'this', 'is', 'an', 'example', 'sentence', 'for', 'token', '##ization', '.', '[SEP]']
input_ids shape: torch.Size([1, 14])


## 2) Embeddings aleatorios para demostracion

In [22]:
# Escribe tu codigo aqui
vocab_size = tokenizar.vocab_size
print("Vocab size:", vocab_size)

embedding_dim = 64  # Dimensión de los embeddings de BERT

embedding_layer = torch.nn.Embedding(vocab_size, embedding_dim)

embedding_output = embedding_layer(encoding["input_ids"])
print("Embedding output shape:", embedding_output.shape)

Vocab size: 30522
Embedding output shape: torch.Size([1, 14, 64])


## 3) Proyecciones Q, K y V

In [ ]:
# Escribe tu codigo aqui
d_model = embedding_dim  # Dimensión del modelo

W_q = torch.nn.Linear(d_model, d_model, bias=False)
W_k = torch.nn.Linear(d_model, d_model, bias=False)
W_v = torch.nn.Linear(d_model, d_model, bias=False)

Q = W_q(embedding_output)
K = W_k(embedding_output) 
V = W_v(embedding_output)

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)

Q shape: torch.Size([1, 14, 64])
K shape: torch.Size([1, 14, 64])
V shape: torch.Size([1, 14, 64])


## 4) Scaled Dot-Product Attention desde cero

In [24]:
# Escribe tu codigo aqui
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)
    return output, weights

salida, pesos = scaled_dot_product_attention(Q, K, V)
print("Output shape:", salida.shape)
print("Attention weights shape:", pesos.shape)

Output shape: torch.Size([1, 14, 64])
Attention weights shape: torch.Size([1, 14, 14])


## 5) Intuicion Q/K/V con 12 frases de ejemplo

In [25]:
# Escribe tu codigo aqui
frases_qkv = [
    "The cat chases the mouse",
    "Maria told Ana that she won",
    "Transformers use attention",
    "Python is a programming language",
    "Self-attention computes weights",
    "The dog barks in the garden",
    "OpenAI released GPT",
    "BERT masks random tokens",
    "Machine translation improved with attention",
    "The CLS token summarizes the sentence",
    "Embeddings capture semantics",
    "The model learns contextual relationships"
    ]

resumen = []

for frase in frases_qkv:
    encoding = tokenizar(frase, return_tensors="pt")['input_ids']
    embedding_output = tokenizar.convert_ids_to_tokens(encoding[0])
    
    resumen.append({
        "frase": frase,
        "tokens": embedding_output,
        "num_tokens": len(embedding_output)
    })

df_resumen = pd.DataFrame(resumen)
display(df_resumen)


,frase,tokens,num_tokens
0,The cat chases the mouse,"[[CLS], the, cat, chases, the, mouse, [SEP]]",7
1,Maria told Ana that she won,"[[CLS], maria, told, ana, that, she, won, [SEP]]",8
2,Transformers use attention,"[[CLS], transformers, use, attention, [SEP]]",5
3,Python is a programming language,"[[CLS], python, is, a, programming, language, ...",7
4,Self-attention computes weights,"[[CLS], self, -, attention, compute, ##s, weig...",8
5,The dog barks in the garden,"[[CLS], the, dog, bark, ##s, in, the, garden, ...",9
6,OpenAI released GPT,"[[CLS], open, ##ai, released, gp, ##t, [SEP]]",7
7,BERT masks random tokens,"[[CLS], bert, masks, random, token, ##s, [SEP]]",7
8,Machine translation improved with attention,"[[CLS], machine, translation, improved, with, ...",7
9,The CLS token summarizes the sentence,"[[CLS], the, cl, ##s, token, sum, ##mar, ##ize...",11


## 6) Interpretacion de pesos de atencion en una frase

In [27]:
# Tomamos una frase corta y mostramos pesos token-token
frase_corta = 'The cat chases the mouse'
enc = tokenizar(frase_corta, return_tensors='pt')
emb = embedding_layer(enc['input_ids'])
q, k, v = W_q(emb), W_k(emb), W_v(emb)
_, attn = scaled_dot_product_attention(q, k, v)

tokens = tokenizar.convert_ids_to_tokens(enc['input_ids'][0])
matriz = pd.DataFrame(attn[0].detach().numpy(), index=tokens, columns=tokens)
display(matriz.round(3))

,[CLS],the,cat,chases,the,mouse,[SEP]
[CLS],0.114,0.193,0.209,0.085,0.193,0.099,0.107
the,0.214,0.094,0.167,0.122,0.094,0.160,0.149
cat,0.100,0.110,0.160,0.099,0.110,0.273,0.147
chases,0.102,0.160,0.114,0.105,0.160,0.200,0.159
the,0.214,0.094,0.167,0.122,0.094,0.160,0.149
mouse,0.172,0.125,0.116,0.115,0.125,0.178,0.169
[SEP],0.130,0.168,0.169,0.142,0.168,0.114,0.109


## Resultados
Tokenizamos texto, generamos embeddings, proyectamos Q/K/V e implementamos atencion escalada sin modelos preentrenados.

## Conclusiones
La Self-Attention permite que cada token consulte a todos los demas. Escalar por sqrt(d_k) estabiliza los gradientes cuando la dimension crece.

## Ejercicios guiados resueltos
**Ejercicio:** Calcula la suma por fila de la matriz de atencion y verifica que sea 1.

**Solucion:**

In [ ]:
# Escribe tu codigo aqui


## Ejercicios propuestos
1. Implementa una mascara causal para un decoder.
2. Compara BPE vs WordPiece en una misma frase.
3. Visualiza heatmap de `matriz` con seaborn.

## Preguntas de reflexion
1. Por que Q, K y V usan proyecciones distintas?
2. Que ocurre si omites la division por sqrt(d_k)?
3. Como cambia el numero de tokens al usar subpalabras?